# R1: SO-ARM101 Reaching タスク — カスタム報酬設計の比較実験

Isaac Sim + Isaac Lab 環境で SO-ARM101 ロボットアームの Reaching タスクを強化学習で解き、
報酬設計の違いによる学習結果の変化を比較する実験ノートです。

## 実験概要

1. **ベースライン**: isaac_so_arm101 のデフォルト報酬で学習
2. **Custom A**: 報酬パラメータの調整のみ（weight / std の変更）
3. **Custom B**: Custom A + カスタム報酬関数（関節リミット回避）の追加

## 前提環境

| 項目 | バージョン |
|------|------------|
| マシン | NVIDIA DGX Spark（GB10 Grace Blackwell） |
| OS | Ubuntu 24.04 ARM64 |
| Isaac Sim | 5.1.0-rc.19（ソースビルド） |
| Isaac Lab | 0.54.3 |
| isaac_so_arm101 | v1.2.0 |
| RL Framework | RSL-RL (PPO) |

環境構築手順は記事を参照してください:
- [DGX Spark でロボットアームの強化学習を試してみた（Isaac Sim + Isaac Lab + SO-ARM101）](https://dev.classmethod.jp/articles/dgx-spark-isaac-sim-so-arm101/)
- [DGX Spark Isaac Playbook](https://build.nvidia.com/spark/isaac/overview)

## 0. 環境設定

パスは環境変数で設定します。デフォルト値は DGX Spark での一般的な配置です。
自分の環境に合わせて変更してください。

In [ ]:
import os
import subprocess
from pathlib import Path

# --- Paths (override with environment variables if needed) ---
ISAAC_SIM_PATH = Path(os.environ.get("ISAACSIM_PATH", str(Path.home() / "IsaacSim")))
ISAAC_LAB_PATH = Path(os.environ.get("ISAACLAB_PATH", str(Path.home() / "IsaacLab")))
SO_ARM101_PATH = Path(os.environ.get("SO_ARM101_PATH", str(Path.home() / "isaac_so_arm101")))

ISAACLAB_SH = str(ISAAC_LAB_PATH / "isaaclab.sh")
TRAIN_SCRIPT = str(
    SO_ARM101_PATH / "src/isaac_so_arm101/scripts/rsl_rl/train.py"
)
PLAY_SCRIPT = str(
    SO_ARM101_PATH / "src/isaac_so_arm101/scripts/rsl_rl/play.py"
)

# --- Environment variables (DGX Spark aarch64) ---
# LD_PRELOAD is required to avoid OpenMP crash on aarch64
os.environ["LD_PRELOAD"] = "/lib/aarch64-linux-gnu/libgomp.so.1"
# DISPLAY is required even in headless mode (Isaac Sim calls XOpenDisplay internally)
os.environ.setdefault("DISPLAY", ":0")

# --- Training parameters ---
NUM_ENVS = 64
MAX_ITERATIONS = 1000

print(f"Isaac Sim:       {ISAAC_SIM_PATH}")
print(f"Isaac Lab:       {ISAAC_LAB_PATH}")
print(f"isaac_so_arm101: {SO_ARM101_PATH}")
print(f"Num envs:        {NUM_ENVS}")
print(f"Max iterations:  {MAX_ITERATIONS}")

In [ ]:
def run_isaaclab(
    args: list[str], timeout: int = 1800
) -> subprocess.CompletedProcess:
    """Run a command via isaaclab.sh and return the result.

    Args:
        args: Arguments to pass after `isaaclab.sh -p`.
        timeout: Timeout in seconds (default: 30 min).
    """
    cmd = [ISAACLAB_SH, "-p"] + args
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=str(ISAAC_LAB_PATH),
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if result.returncode != 0:
        print(f"STDERR (last 2000 chars):\n{result.stderr[-2000:]}")
    return result

## 1. 事前準備: カスタム環境の登録

Custom A / Custom B の実験には、カスタム報酬環境の登録が必要です。
このリポジトリの `scripts/r1-custom-reach-env.py` を isaac_so_arm101 パッケージにコピーし、
`__init__.py` に import を追加します。

```bash
# 1. Copy custom env definition
cp ../scripts/r1-custom-reach-env.py \
   ${SO_ARM101_PATH}/src/isaac_so_arm101/r1_custom_reach_env.py

# 2. Register custom envs (add import to __init__.py)
echo "import isaac_so_arm101.r1_custom_reach_env" >> \
   ${SO_ARM101_PATH}/src/isaac_so_arm101/__init__.py
```

登録されるカスタム環境:

| Env ID | 変更内容 |
|--------|----------|
| `Isaac-SO-ARM101-Reach-CustomA-v0` | position penalty -0.2→-0.5, tanh std 0.1→0.05, action_rate -0.0001→-0.001 |
| `Isaac-SO-ARM101-Reach-CustomB-v0` | Custom A + joint limit avoidance reward (weight=0.05, std=0.2) |

In [ ]:
# Verify custom env registration
import shutil

custom_env_src = Path("../scripts/r1-custom-reach-env.py")
custom_env_dst = SO_ARM101_PATH / "src/isaac_so_arm101/r1_custom_reach_env.py"

if not custom_env_dst.exists():
    print(f"Copying {custom_env_src} -> {custom_env_dst}")
    shutil.copy2(custom_env_src, custom_env_dst)

    # Add import to __init__.py
    init_py = SO_ARM101_PATH / "src/isaac_so_arm101/__init__.py"
    import_line = "import isaac_so_arm101.r1_custom_reach_env\n"
    content = init_py.read_text()
    if "r1_custom_reach_env" not in content:
        with open(init_py, "a") as f:
            f.write(import_line)
        print(f"Added import to {init_py}")
    print("Custom environments registered.")
else:
    print(f"Custom env already exists at {custom_env_dst}")

## 2. ベースライン学習

isaac_so_arm101 のデフォルト報酬設定で Reaching タスクを学習します。

### 報酬構成

| 報酬項 | 関数 | weight | 説明 |
|--------|------|--------|------|
| position tracking | `position_command_error` | -0.2 | L2 距離ペナルティ（大まかな接近） |
| fine-grained tracking | `position_command_error_tanh` | 0.1 (std=0.1) | tanh カーネルによる近距離ボーナス |
| action rate | `action_rate_l2` | -0.0001 | アクション変化率ペナルティ |
| joint velocity | `joint_vel_l2` | -0.0001 | 関節速度ペナルティ |

カリキュラム学習で action_rate → -0.005, joint_vel → -0.001 に 4500 steps かけて強化されます。

In [ ]:
baseline_result = run_isaaclab([
    TRAIN_SCRIPT,
    "--task", "Isaac-SO-ARM101-Reach-v0",
    "--headless",
    "--num_envs", str(NUM_ENVS),
    "--max_iterations", str(MAX_ITERATIONS),
])
print(baseline_result.stdout[-3000:])

### ベースライン評価

学習済みポリシーで動画を記録します。タスク名は末尾に `-Play` を付けてください。

In [ ]:
baseline_play = run_isaaclab([
    PLAY_SCRIPT,
    "--task", "Isaac-SO-ARM101-Reach-Play-v0",
    "--num_envs", "4",
    "--video",
    "--video_length", "200",
])
print(baseline_play.stdout[-2000:])

## 3. Custom A — パラメータ調整

既存の報酬関数はそのままに、weight と std を調整します。

| パラメータ | Baseline | Custom A |
|-----------|----------|----------|
| `position_command_error` weight | -0.2 | **-0.5** |
| `position_command_error_tanh` std | 0.1 | **0.05** |
| `action_rate` weight | -0.0001 | **-0.001** |

距離ペナルティの強化、tanh 感度の引き上げ、序盤からの動作平滑化を狙います。

In [ ]:
custom_a_result = run_isaaclab([
    TRAIN_SCRIPT,
    "--task", "Isaac-SO-ARM101-Reach-CustomA-v0",
    "--headless",
    "--num_envs", str(NUM_ENVS),
    "--max_iterations", str(MAX_ITERATIONS),
])
print(custom_a_result.stdout[-3000:])

## 4. Custom B — カスタム報酬関数の追加

Custom A のパラメータ調整に加え、**関節リミット回避報酬**を追加します。

```python
def joint_pos_limit_avoidance(env, std, asset_cfg):
    """各関節が可動域リミットから離れているほど高い報酬を返す。"""
    # tanh(dist_to_nearest_limit / std)
    # リミットから遠い → ~1.0, リミット付近（0.2 rad ≒ 11 度以内） → ~0.0
```

| パラメータ | 値 |
|-----------|-----|
| weight | 0.05 |
| std | 0.2 rad |

関節リミット回避が正則化として機能し、可動域中央付近での安定した動作を促す狙いです。
実装の詳細は `../scripts/r1-custom-reach-env.py` を参照してください。

In [ ]:
custom_b_result = run_isaaclab([
    TRAIN_SCRIPT,
    "--task", "Isaac-SO-ARM101-Reach-CustomB-v0",
    "--headless",
    "--num_envs", str(NUM_ENVS),
    "--max_iterations", str(MAX_ITERATIONS),
])
print(custom_b_result.stdout[-3000:])

## 5. 結果比較

TensorBoard のログから主要メトリクスを抽出し、3 パターンを比較します。

> **Note**: matplotlib は Isaac Sim Python 3 カーネルでは動作しません。
> 可視化セルは標準の Python カーネルで実行するか、
> TensorBoard (`tensorboard --logdir logs/`) で直接確認してください。

In [ ]:
# TensorBoard log parsing helper
try:
    from tensorboard.backend.event_processing.event_accumulator import (
        EventAccumulator,
    )
    import pandas as pd

    def load_tb_scalar(logdir: str, tag: str) -> pd.DataFrame:
        """Load a scalar tag from TensorBoard event logs."""
        ea = EventAccumulator(logdir)
        ea.Reload()
        events = ea.Scalars(tag)
        return pd.DataFrame(
            [(e.step, e.value) for e in events],
            columns=["step", "value"],
        )

    print("TensorBoard event loading ready.")
except ImportError:
    print("tensorboard or pandas not installed.")
    print("Install with: pip install tensorboard pandas")

In [ ]:
# --- Set your actual log directories here ---
# After training, check the output for the log directory path.
# Example:
#   LOG_BASELINE = "logs/rsl_rl/Isaac-SO-ARM101-Reach-v0/2026-02-24_10-30-00"
#   LOG_CUSTOM_A = "logs/rsl_rl/Isaac-SO-ARM101-Reach-CustomA-v0/2026-02-24_10-40-00"
#   LOG_CUSTOM_B = "logs/rsl_rl/Isaac-SO-ARM101-Reach-CustomB-v0/2026-02-24_10-50-00"

LOG_BASELINE = ""  # TODO: set after training
LOG_CUSTOM_A = ""  # TODO: set after training
LOG_CUSTOM_B = ""  # TODO: set after training

In [ ]:
# Comparison visualization
try:
    import matplotlib.pyplot as plt

    assert LOG_BASELINE and LOG_CUSTOM_A and LOG_CUSTOM_B, (
        "Set LOG_BASELINE, LOG_CUSTOM_A, LOG_CUSTOM_B first."
    )

    configs = {
        "Baseline": LOG_BASELINE,
        "Custom A": LOG_CUSTOM_A,
        "Custom B": LOG_CUSTOM_B,
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Position Error
    ax = axes[0]
    for name, logdir in configs.items():
        df = load_tb_scalar(logdir, "Metrics/position_error")
        ax.plot(df["step"], df["value"], label=name)
    ax.set_xlabel("Step")
    ax.set_ylabel("Position Error")
    ax.set_title("Position Error")
    ax.legend()

    # Mean Reward
    ax = axes[1]
    for name, logdir in configs.items():
        df = load_tb_scalar(logdir, "Metrics/mean_reward")
        ax.plot(df["step"], df["value"], label=name)
    ax.set_xlabel("Step")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Mean Reward")
    ax.legend()

    plt.tight_layout()
    plt.savefig("position_error_comparison.png", dpi=150)
    plt.show()
    print("Saved: position_error_comparison.png")

except ImportError:
    print("matplotlib not available. Use TensorBoard for visualization:")
    print("  tensorboard --logdir logs/")
except AssertionError as e:
    print(e)

## 6. 結果サマリー

DGX Spark（GB10 GPU）、64 並列環境、1000 イテレーションでの結果です。

| 項目 | Baseline | Custom A（パラメータ調整） | Custom B（関数追加） |
|------|----------|---------------------------|---------------------|
| position error | 0.0987 | 0.1279 | **0.0802** |
| 平均報酬 | 0.23 | -0.72 | 0.03 |
| スループット | 3,514 steps/s | 3,446 steps/s | 3,043 steps/s |
| 所要時間 | 約 9 分 | 約 9 分 | 約 9 分 |

### 考察

- **Custom A**: ペナルティ強化が裏目に出た。距離ペナルティ -0.5 でエージェントが「罰回避」に偏り、
  tanh std=0.05 でボーナス獲得範囲が狭すぎた可能性がある
- **Custom B**: 関節リミット回避が正則化として機能し、可動域中央付近での動作が目標到達効率を改善した。
  `joint_limit_avoidance` が 0.0461 と安定して報酬を獲得しており、学習の安定化に貢献
- **学び**: 「既存ペナルティの強化」より「補助的な報酬の追加」が効果的な場合がある

## 補足

### カーネル選択

この notebook は 2 つのカーネルでの利用を想定しています。

| セクション | 推奨カーネル | 理由 |
|-----------|-------------|------|
| 0-4（学習・評価） | Isaac Sim Python 3 | Isaac Sim API へのアクセスが必要 |
| 5-6（分析・可視化） | 標準 Python 3 | matplotlib が必要 |

Isaac Sim カーネルの設定方法:
1. Isaac Sim の Extension Manager で `isaacsim.code_editor.jupyter` を有効化
2. JupyterLab で `Isaac Sim Python 3` カーネルを選択

詳細: [Isaac Sim Jupyter Notebook ドキュメント](https://docs.isaacsim.omniverse.nvidia.com/latest/development_tools/jupyter_notebook.html)

### トラブルシューティング（DGX Spark aarch64）

| 問題 | 対処 |
|------|------|
| OpenMP クラッシュ | `LD_PRELOAD="/lib/aarch64-linux-gnu/libgomp.so.1"` を設定 |
| SSH 経由でハング | DISPLAY / XAUTHORITY を設定、またはデスクトップ環境で実行 |
| タスク名が見つからない | `Isaac-` プレフィックスを確認（例: `Isaac-SO-ARM101-Reach-v0`） |
| play.py で ImportError | Isaac Lab 0.54.3 に未実装のモジュール。try-except で回避 |